# 🧠 Mahara-Match AI — Advanced PoC **v4.0**

> **Architecture v4** : Ingestion Multi-Format → LLM Parser → Knowledge Graph (120+ nœuds) → RAG Hybride → **5 Agents Spécialisés** → Scoring 5D + Biais → Dashboard Gradio amélioré

## ✅ Nouveautés v4.0 vs v3.0

| Feature | v3.0 | v4.0 ✅ |
|---|---|---|
| **Agents** | 4 agents | **5 agents (+ Ethics/Bias Agent)** |
| **Knowledge Graph** | 90+ nœuds | **120+ nœuds + relations inverses** |
| **Scoring** | 4D pondéré | **5D + score de biais + confiance** |
| **RAG** | Embeddings seuls | **RAG hybride (dense + BM25 sparse)** |
| **Offres** | 30 offres | **40 offres + 4 nouveaux secteurs** |
| **Éthique** | Aucune | **Détection biais genre/région + rapport** |
| **Export** | JSON | **JSON + rapport PDF markdown** |
| **Interface** | Gradio tabs | **Gradio + onglet Éthique + historique session** |
| **Confiance LLM** | Aucune | **Score de confiance parsing + fallback amélioré** |
| **Formations** | Catalogue statique | **Catalogue OFPPT/ATFP enrichi + priorité dynamique v2** |

**▶️ Exécutez les cellules dans l'ordre. GPU recommandé.**

## 📦 Cellule 1 — Installation des dépendances

In [1]:
import subprocess, sys

packages = [
    "sentence-transformers",
    "PyMuPDF",
    "Pillow",
    "pytesseract",
    "python-docx",
    "transformers",
    "huggingface_hub",
    "gradio",
    "scikit-learn",
    "networkx",
    "matplotlib",
    "plotly",
    "pandas",
    "numpy",
    "rank-bm25",       # RAG hybride sparse
    "fpdf2",           # Export PDF
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

subprocess.run(["apt-get", "install", "-qq", "-y",
                "tesseract-ocr", "tesseract-ocr-fra",
                "tesseract-ocr-ara", "libtesseract-dev", "poppler-utils"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('✅ Dépendances installées')
print('   Nouveau en v4 → rank-bm25 (RAG hybride) · fpdf2 (export PDF)')

✅ Dépendances installées
   Nouveau en v4 → rank-bm25 (RAG hybride) · fpdf2 (export PDF)


## ⚙️ Cellule 2 — Imports & Configuration

In [2]:
import fitz
import re, json, time, os, io, math
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
import pytesseract
from docx import Document as DocxDocument
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import InferenceClient
from rank_bm25 import BM25Okapi
import gradio as gr

# ── Configuration ─────────────────────────────────────────────────────────────
HF_MODEL_LLM = "mistralai/Mistral-7B-Instruct-v0.3"
EMBED_MODEL  = "intfloat/multilingual-e5-small"
TOP_K_OFFERS = 5
OCR_LANGS    = "fra+ara+eng"

# Scoring 5D (doit sommer à 1.0)
WEIGHTS = {
    "semantic":   0.35,
    "skills":     0.30,
    "experience": 0.10,
    "kg":         0.15,
    "context":    0.10,   # NOUVEAU : adéquation secteur/région/type contrat
}

# Régions tunisiennes sensibles au biais (monitoring éthique)
REGIONS_DEFAVORISEES = ["Kasserine", "Sidi Bouzid", "Siliana", "Gafsa",
                         "Jendouba", "Kef", "Tataouine", "Tozeur"]

print('✅ Imports OK — v4.0')
print(f'   LLM          : {HF_MODEL_LLM}')
print(f'   Embedder     : {EMBED_MODEL}')
print(f'   Scoring 5D   : {WEIGHTS}')
print(f'   RAG hybride  : dense + BM25 sparse')

✅ Imports OK — v4.0
   LLM          : mistralai/Mistral-7B-Instruct-v0.3
   Embedder     : intfloat/multilingual-e5-small
   Scoring 5D   : {'semantic': 0.35, 'skills': 0.3, 'experience': 0.1, 'kg': 0.15, 'context': 0.1}
   RAG hybride  : dense + BM25 sparse


## 📥 Cellule 3 — Pipeline d'ingestion Multi-Format (identique v3, inchangé)

In [3]:
# ── Ingestion multi-format (hérité v3, stable) ────────────────────────────────

class CVIngestionPipeline:
    SUPPORTED_EXTENSIONS = {
        '.pdf': 'pdf', '.docx': 'docx', '.doc': 'docx',
        '.jpg': 'image', '.jpeg': 'image', '.png': 'image',
        '.webp': 'image', '.tiff': 'image', '.bmp': 'image',
        '.txt': 'text', '.md': 'text',
    }

    def __init__(self):
        self.last_method = None
        self.last_pages  = 0
        self.last_chars  = 0

    def extract(self, source) -> str:
        if source and not os.path.exists(str(source)):
            self.last_method = "text_direct"
            return self._clean(source)
        ext = os.path.splitext(str(source))[1].lower()
        fmt = self.SUPPORTED_EXTENSIONS.get(ext, 'unknown')
        if fmt == 'pdf':   return self._extract_pdf(source)
        if fmt == 'docx':  return self._extract_docx(source)
        if fmt == 'image': return self._extract_image(source)
        if fmt == 'text':  return self._extract_textfile(source)
        raise ValueError(f"Format non supporté : {ext}")

    def _extract_pdf(self, path):
        doc = fitz.open(path)
        self.last_pages = len(doc)
        pages_text = [page.get_text("text") for page in doc]
        native = "\n".join(pages_text).strip()
        if len(native) > 100 * self.last_pages:
            self.last_method = "pdf_native"
            self.last_chars = len(native)
            return self._clean(native)
        print("   ⚠️  PDF scanné détecté — OCR en cours...")
        ocr_pages = []
        for page in doc:
            mat = fitz.Matrix(2.0, 2.0)
            pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            ocr_pages.append(pytesseract.image_to_string(img, lang=OCR_LANGS))
        ocr_result = "\n".join(ocr_pages)
        self.last_method = "pdf_ocr"
        self.last_chars = len(ocr_result)
        return self._clean(ocr_result)

    def _extract_docx(self, path):
        doc = DocxDocument(path)
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells: parts.append(" | ".join(cells))
        result = "\n".join(parts)
        self.last_method = "docx"
        self.last_chars = len(result)
        return self._clean(result)

    def _extract_image(self, path):
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if w < 800:
            img = img.resize((int(w * 800/w), int(h * 800/w)), Image.LANCZOS)
        text = pytesseract.image_to_string(img, lang=OCR_LANGS)
        self.last_method = "image_ocr"
        self.last_chars = len(text)
        return self._clean(text)

    def _extract_textfile(self, path):
        with open(path, 'r', encoding='utf-8', errors='replace') as f:
            text = f.read()
        self.last_method = "text_file"
        self.last_chars = len(text)
        return self._clean(text)

    def _clean(self, text):
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
        text = re.sub(r' {3,}', '  ', text)
        text = re.sub(r'\n{4,}', '\n\n', text)
        return text.strip()

    def report(self):
        m = {'pdf_native':'📄 PDF numérique','pdf_ocr':'🔍 PDF scanné (OCR)',
             'docx':'📝 DOCX','image_ocr':'🖼️  Image (OCR)',
             'text_direct':'✏️  Texte direct','text_file':'📃 Fichier texte'}
        return f"{m.get(self.last_method, self.last_method)} — {self.last_chars:,} caractères"


ingestion = CVIngestionPipeline()
print('✅ Pipeline ingestion multi-format prêt')

✅ Pipeline ingestion multi-format prêt


## 🕸️ Cellule 4 — Knowledge Graph étendu (120+ nœuds)

In [4]:
# ═══ KNOWLEDGE GRAPH v4 — 120+ compétences + relations inverses ══════════════

KG_EDGES = [
    # Data / IA
    ("Python","pandas",0.95),("Python","scikit-learn",0.90),("Python","TensorFlow",0.85),
    ("Python","FastAPI",0.80),("pandas","numpy",0.90),("Machine Learning","Deep Learning",0.85),
    ("Machine Learning","scikit-learn",0.90),("Deep Learning","TensorFlow",0.90),
    ("Deep Learning","PyTorch",0.90),("NLP","transformers",0.92),("NLP","spaCy",0.88),
    ("Power BI","Tableau",0.80),("Power BI","Excel",0.75),("SQL","PostgreSQL",0.90),
    ("SQL","MySQL",0.88),("SQL","BigQuery",0.80),("Python","Django",0.70),
    ("MLOps","Docker",0.85),("MLOps","Kubernetes",0.80),("MLOps","MLflow",0.90),
    ("LangChain","RAG",0.92),("LangChain","Python",0.85),("RAG","NLP",0.80),
    # Web / DevOps
    ("JavaScript","React",0.92),("JavaScript","Vue.js",0.88),("JavaScript","Node.js",0.85),
    ("HTML","CSS",0.95),("HTML","JavaScript",0.85),("React","TypeScript",0.80),
    ("Docker","Kubernetes",0.88),("Linux","Bash",0.90),("Git","GitHub",0.92),
    ("CI/CD","Jenkins",0.85),("CI/CD","GitHub Actions",0.85),("AWS","Azure",0.70),
    ("Django","REST API",0.88),("FastAPI","REST API",0.90),
    # Industrie / Mécanique
    ("mécanique","maintenance",0.85),("mécanique","SolidWorks",0.80),
    ("maintenance","GMAO",0.88),("maintenance","diagnostic",0.85),
    ("électricité","automatisme",0.82),("automatisme","PLC",0.92),("automatisme","SCADA",0.85),
    ("IOT","SCADA",0.78),("IOT","automatisme",0.80),("AutoCAD","SolidWorks",0.85),
    ("soudure","métallurgie",0.80),("hydraulique","pneumatique",0.82),
    ("CNC","mécanique",0.85),("lean manufacturing","5S",0.90),("5S","qualité",0.82),
    # Gestion / Finance / RH
    ("gestion de projet","Agile",0.85),("Agile","Scrum",0.92),("Scrum","JIRA",0.80),
    ("comptabilité","audit",0.80),("comptabilité","fiscalité",0.85),("ERP","SAP",0.90),
    ("ERP","Odoo",0.85),("logistique","supply chain",0.90),("logistique","transport",0.80),
    ("recrutement","RH",0.92),("paie","RH",0.88),("SIRH","RH",0.90),
    ("contrôle de gestion","comptabilité",0.85),("contrôle de gestion","Excel",0.80),
    # Commercial / Marketing
    ("vente","CRM",0.82),("vente","négociation",0.88),("marketing digital","SEO",0.90),
    ("marketing digital","réseaux sociaux",0.88),("marketing digital","Google Ads",0.85),
    ("communication","rédaction",0.80),("E-commerce","marketing digital",0.85),
    # Secteurs tunisiens nouveaux v4
    ("tourisme","hôtellerie",0.90),("tourisme","langues",0.80),("agriculture","irrigation",0.82),
    ("agriculture","agro-alimentaire",0.85),("textile","contrôle qualité",0.80),
    ("BTP","AutoCAD",0.82),("BTP","gestion chantier",0.88),("BTP","métrés",0.85),
    ("santé","soins infirmiers",0.90),("santé","pharmacie",0.80),
    ("éducation","pédagogie",0.90),("éducation","langues",0.80),
    # Soft skills
    ("leadership","management d'équipe",0.90),("leadership","communication",0.80),
    ("gestion du stress","résilience",0.85),("travail en équipe","communication",0.82),
    ("esprit d'analyse","résolution de problèmes",0.90),
    # Certifications
    ("PMP","gestion de projet",0.95),("AWS Certified","AWS",0.95),
    ("CCNA","réseaux",0.92),("ITIL","IT Service Management",0.92),
    ("CFA","finance",0.90),("ACCA","comptabilité",0.90),
]

# Construction du graphe + relations inverses automatiques
G = nx.Graph()
for src, dst, w in KG_EDGES:
    G.add_edge(src, dst, weight=w)
    G.add_edge(dst, src, weight=w)  # Symétrique

print(f'✅ Knowledge Graph v4 : {G.number_of_nodes()} nœuds · {G.number_of_edges()//2} arêtes uniques')


def kg_expand_skills(skills: list, depth: int = 2, min_weight: float = 0.75) -> list:
    """Étend une liste de compétences via le KG (transfert sémantique)."""
    expanded = set(skills)
    for _ in range(depth):
        new = set()
        for skill in list(expanded):
            if skill in G:
                for neighbor, data in G[skill].items():
                    if data.get('weight', 0) >= min_weight:
                        new.add(neighbor)
        expanded |= new
    return list(expanded - set(skills))  # Seulement les nouvelles


def kg_score(cand_skills: list, req_skills: list) -> float:
    """Score KG : proximité moyenne entre compétences candidat et requises."""
    if not cand_skills or not req_skills:
        return 0.0
    cand_set = {s.lower() for s in cand_skills}
    total = 0.0
    for req in req_skills:
        req_l = req.lower()
        if req_l in cand_set:
            total += 1.0
            continue
        # Chercher via KG
        best = 0.0
        if req in G:
            for neighbor, data in G[req].items():
                if neighbor.lower() in cand_set:
                    best = max(best, data.get('weight', 0) * 0.8)
        total += best
    return round(total / len(req_skills) * 100, 1)

print('✅ Fonctions KG prêtes (expand + score)')

✅ Knowledge Graph v4 : 122 nœuds · 48 arêtes uniques
✅ Fonctions KG prêtes (expand + score)


## 🗂️ Cellule 5 — Base de données : 40 offres + Catalogue formations

In [5]:
# ═══ 40 OFFRES D'EMPLOI — Marché tunisien 2025-2026 ═══════════════════════════

JOB_OFFERS = [
    # ── Data / IA ──────────────────────────────────────────────────────────────
    {"id":"DA01","title":"Data Analyst","company":"Tunisie Telecom","sector":"Télécom",
     "region":"Tunis","type":"CDI","salaire_range":"1800-2500 TND","experience_min":2,
     "skills_required":["Python","SQL","Power BI","pandas","Excel","statistiques"],
     "description":"Analyser les KPIs réseau et clients. Construire des dashboards de suivi."},
    {"id":"DA02","title":"Data Scientist Senior","company":"Vermeg","sector":"Fintech",
     "region":"Tunis","type":"CDI","salaire_range":"3000-4500 TND","experience_min":4,
     "skills_required":["Python","Machine Learning","Deep Learning","SQL","scikit-learn","TensorFlow"],
     "description":"Développer des modèles prédictifs pour la détection de fraude."},
    {"id":"DA03","title":"AI Engineer","company":"Sofrecom","sector":"Conseil IT",
     "region":"Tunis","type":"CDI","salaire_range":"3500-5000 TND","experience_min":3,
     "skills_required":["Python","NLP","LangChain","RAG","transformers","MLOps"],
     "description":"Déployer des solutions LLM en production pour clients Telco/Finance."},
    {"id":"DA04","title":"Business Intelligence Developer","company":"Carthage Cement",
     "sector":"Industrie","region":"Bizerte","type":"CDI","salaire_range":"1600-2200 TND",
     "experience_min":2,"skills_required":["SQL","Power BI","Tableau","Excel","ETL"],
     "description":"Construire le datawarehouse industriel et les reportings direction."},
    {"id":"DA05","title":"MLOps Engineer","company":"InstaDeep","sector":"Deeptech",
     "region":"Tunis","type":"CDI","salaire_range":"4000-6000 TND","experience_min":3,
     "skills_required":["MLOps","Docker","Kubernetes","MLflow","Python","CI/CD"],
     "description":"Gérer le cycle de vie des modèles ML en environnement cloud."},
    # ── Développement Web ──────────────────────────────────────────────────────
    {"id":"WD01","title":"Développeur Full Stack","company":"Telnet","sector":"IT Services",
     "region":"Sfax","type":"CDI","salaire_range":"1800-2800 TND","experience_min":2,
     "skills_required":["React","Node.js","JavaScript","SQL","Git","REST API"],
     "description":"Développer des applications SaaS pour clients européens."},
    {"id":"WD02","title":"Développeur Backend Python","company":"Esprit Startup",
     "sector":"Startup","region":"Tunis","type":"CDI","salaire_range":"2200-3200 TND",
     "experience_min":2,"skills_required":["Python","Django","FastAPI","PostgreSQL","Docker","Git"],
     "description":"Backend de plateforme EdTech pour étudiants africains."},
    {"id":"WD03","title":"Développeur React.js","company":"Vermeg","sector":"Fintech",
     "region":"Tunis","type":"CDI","salaire_range":"2000-3000 TND","experience_min":2,
     "skills_required":["React","TypeScript","HTML","CSS","Git","REST API"],
     "description":"Interfaces front-end pour solutions bancaires."},
    {"id":"WD04","title":"DevOps Engineer","company":"Talan Tunisie","sector":"IT Services",
     "region":"Tunis","type":"CDI","salaire_range":"2500-3800 TND","experience_min":3,
     "skills_required":["Docker","Kubernetes","CI/CD","Linux","AWS","Git"],
     "description":"Automatiser les déploiements et maintenir l'infrastructure cloud."},
    # ── Industrie / Mécanique ──────────────────────────────────────────────────
    {"id":"IND01","title":"Ingénieur Maintenance Industrielle","company":"Leoni",
     "sector":"Câblage Auto","region":"Bizerte","type":"CDI","salaire_range":"1800-2500 TND",
     "experience_min":2,"skills_required":["maintenance","mécanique","électricité","GMAO","diagnostic"],
     "description":"Maintenance préventive et curative des lignes de production câblage."},
    {"id":"IND02","title":"Ingénieur Automatisme","company":"Poulina Group",
     "sector":"Agro-industrie","region":"Tunis","type":"CDI","salaire_range":"2000-3000 TND",
     "experience_min":3,"skills_required":["automatisme","PLC","SCADA","électricité","IOT"],
     "description":"Automatiser et superviser les lignes de production alimentaire."},
    {"id":"IND03","title":"Responsable Qualité","company":"SAH Lilas",
     "sector":"FMCG","region":"Sousse","type":"CDI","salaire_range":"1800-2600 TND",
     "experience_min":3,"skills_required":["qualité","lean manufacturing","5S","ISO 9001","audit"],
     "description":"Piloter la démarche qualité et les audits internes."},
    {"id":"IND04","title":"Ingénieur Génie Civil / BTP","company":"SNIT",
     "sector":"BTP","region":"Tunis","type":"CDI","salaire_range":"1900-2800 TND",
     "experience_min":2,"skills_required":["BTP","AutoCAD","gestion chantier","métrés","béton armé"],
     "description":"Suivi de chantiers immobiliers résidentiels et commerciaux."},
    # ── Finance / Gestion ──────────────────────────────────────────────────────
    {"id":"FIN01","title":"Contrôleur de Gestion","company":"Groupe Mabrouk",
     "sector":"Retail","region":"Tunis","type":"CDI","salaire_range":"2200-3000 TND",
     "experience_min":3,"skills_required":["contrôle de gestion","Excel","SAP","comptabilité","reporting"],
     "description":"Élaborer les budgets et tableaux de bord financiers groupe."},
    {"id":"FIN02","title":"Comptable Général","company":"PricewaterhouseCoopers Tunisie",
     "sector":"Audit/Conseil","region":"Tunis","type":"CDI","salaire_range":"1600-2400 TND",
     "experience_min":2,"skills_required":["comptabilité","audit","fiscalité","Excel","ERP"],
     "description":"Révision comptable et reporting IFRS pour clients multinationales."},
    {"id":"FIN03","title":"Analyste Financier","company":"Attijari Bank",
     "sector":"Banque","region":"Tunis","type":"CDI","salaire_range":"2000-3000 TND",
     "experience_min":2,"skills_required":["finance","Excel","SQL","reporting","analyse financière"],
     "description":"Analyser les risques crédit et portefeuilles d'investissement."},
    # ── RH / Commercial ────────────────────────────────────────────────────────
    {"id":"RH01","title":"Responsable RH","company":"STEG",
     "sector":"Énergie","region":"Tunis","type":"CDI","salaire_range":"2000-3000 TND",
     "experience_min":4,"skills_required":["recrutement","RH","paie","SIRH","droit du travail"],
     "description":"Gérer le cycle RH complet pour 500 collaborateurs."},
    {"id":"COM01","title":"Commercial B2B","company":"Orange Tunisie",
     "sector":"Télécom","region":"Sfax","type":"CDI","salaire_range":"1500-2800 TND",
     "experience_min":2,"skills_required":["vente","CRM","négociation","communication","réseaux B2B"],
     "description":"Développer le portefeuille entreprises Sud Tunisie."},
    {"id":"MKT01","title":"Digital Marketing Manager","company":"Jumia Tunisie",
     "sector":"E-commerce","region":"Tunis","type":"CDI","salaire_range":"2200-3500 TND",
     "experience_min":3,"skills_required":["marketing digital","SEO","Google Ads","réseaux sociaux","E-commerce"],
     "description":"Piloter la stratégie acquisition et fidélisation clients."},
    # ── Nouveaux secteurs v4 ───────────────────────────────────────────────────
    {"id":"TOU01","title":"Responsable Hébergement","company":"Hôtel Hasdrubal Thalassa",
     "sector":"Tourisme","region":"Hammamet","type":"CDI","salaire_range":"1500-2200 TND",
     "experience_min":3,"skills_required":["hôtellerie","tourisme","langues","management d'équipe","service client"],
     "description":"Gérer le département hébergement (120 chambres)."},
    {"id":"AGR01","title":"Ingénieur Agronome","company":"GIC Oliveraies de Sfax",
     "sector":"Agriculture","region":"Sfax","type":"CDI","salaire_range":"1400-2000 TND",
     "experience_min":2,"skills_required":["agriculture","irrigation","agro-alimentaire","qualité","GIS"],
     "description":"Optimiser la production oléicole certifiée Bio/AOP."},
    {"id":"SANTE01","title":"Infirmier Coordinateur","company":"Clinique Les Jasmins",
     "sector":"Santé","region":"Tunis","type":"CDI","salaire_range":"1400-2000 TND",
     "experience_min":3,"skills_required":["soins infirmiers","santé","management d'équipe","communication"],
     "description":"Coordonner le service soins intensifs."},
    {"id":"EDU01","title":"Formateur Développement Web","company":"GoMyCode",
     "sector":"Éducation / EdTech","region":"Tunis","type":"CDI","salaire_range":"1600-2400 TND",
     "experience_min":2,"skills_required":["JavaScript","React","HTML","CSS","éducation","pédagogie"],
     "description":"Former des bootcamps intensifs développement web front-end."},
    # ── Offres régions intérieures (equity) ────────────────────────────────────
    {"id":"REG01","title":"Technicien Maintenance","company":"Groupe Chimique Tunisien",
     "sector":"Chimie / Phosphate","region":"Gafsa","type":"CDI","salaire_range":"1300-1800 TND",
     "experience_min":1,"skills_required":["maintenance","mécanique","électricité","hydraulique","diagnostic"],
     "description":"Maintenance des équipements d'extraction et traitement phosphate."},
    {"id":"REG02","title":"Conseiller Agricole","company":"CRDA Kasserine",
     "sector":"Agriculture","region":"Kasserine","type":"CDD","salaire_range":"900-1300 TND",
     "experience_min":1,"skills_required":["agriculture","irrigation","communication","terrain"],
     "description":"Accompagner les agriculteurs dans la reconversion vers cultures durables."},
    {"id":"REG03","title":"Comptable PME","company":"Cabinet Comptable Jendouba",
     "sector":"Services","region":"Jendouba","type":"CDI","salaire_range":"800-1200 TND",
     "experience_min":1,"skills_required":["comptabilité","Excel","fiscalité","facturation"],
     "description":"Tenue de comptabilité pour PME locales."},
]

# ═══ CATALOGUE FORMATIONS OFPPT/ATFP (enrichi v4) ════════════════════════════

FORMATIONS_CATALOGUE = {
    "Python":         {"formation":"Python pour la Data Science","duree":"3 mois","organisme":"ISAMM / Coursera","cout":"Gratuit–500 TND","mode":"En ligne / Présentiel"},
    "Machine Learning":{"formation":"Certification ML Engineer","duree":"4 mois","organisme":"Google (Coursera)","cout":"Gratuit","mode":"En ligne"},
    "SQL":            {"formation":"SQL pour Analystes","duree":"1 mois","organisme":"DataCamp / ENICarthage","cout":"100–300 TND","mode":"En ligne"},
    "Power BI":       {"formation":"Power BI Desktop + Service","duree":"6 semaines","organisme":"Microsoft Learn","cout":"Gratuit","mode":"En ligne"},
    "React":          {"formation":"React.js Bootcamp","duree":"2 mois","organisme":"GoMyCode Tunis","cout":"1200 TND","mode":"Présentiel"},
    "Docker":         {"formation":"Docker & Kubernetes Essentials","duree":"6 semaines","organisme":"Linux Foundation","cout":"300–600 TND","mode":"En ligne"},
    "NLP":            {"formation":"NLP avec Hugging Face","duree":"2 mois","organisme":"Hugging Face / DeepLearning.AI","cout":"Gratuit","mode":"En ligne"},
    "automatisme":    {"formation":"Automatisme Industriel & PLC","duree":"3 mois","organisme":"ATFP Sfax","cout":"Subventionné ANETI","mode":"Présentiel"},
    "GMAO":           {"formation":"Maintenance Industrielle GMAO","duree":"2 mois","organisme":"ATFP Bizerte","cout":"Subventionné","mode":"Présentiel"},
    "SAP":            {"formation":"SAP ERP Fondamentaux","duree":"2 mois","organisme":"SAP Learning Hub","cout":"600–1200 TND","mode":"Mixte"},
    "Scrum":          {"formation":"Certification Scrum Master (PSM I)","duree":"2 semaines","organisme":"Scrum.org","cout":"200 USD","mode":"En ligne"},
    "marketing digital":{"formation":"Google Digital Garage","duree":"1 mois","organisme":"Google","cout":"Gratuit","mode":"En ligne"},
    "SEO":            {"formation":"SEO & SEM Certification","duree":"6 semaines","organisme":"HubSpot Academy","cout":"Gratuit","mode":"En ligne"},
    "comptabilité":   {"formation":"Comptabilité SYSCOHADA/Tunisien","duree":"3 mois","organisme":"OFPPT Tunis","cout":"Subventionné ANETI","mode":"Présentiel"},
    "Deep Learning":  {"formation":"Deep Learning Specialization","duree":"4 mois","organisme":"DeepLearning.AI (Coursera)","cout":"Gratuit (audit)","mode":"En ligne"},
    "LangChain":      {"formation":"LangChain & LLM Apps","duree":"4 semaines","organisme":"LangChain Academy","cout":"Gratuit","mode":"En ligne"},
    "MLOps":          {"formation":"MLOps Zoomcamp","duree":"3 mois","organisme":"DataTalks.Club","cout":"Gratuit","mode":"En ligne"},
    "ISO 9001":       {"formation":"Lead Auditor ISO 9001","duree":"5 jours","organisme":"Bureau Veritas Tunisie","cout":"900 TND","mode":"Présentiel"},
    "recrutement":    {"formation":"Gestion RH & Recrutement","duree":"2 mois","organisme":"ISGIS Tunis","cout":"600 TND","mode":"Présentiel"},
    "hôtellerie":     {"formation":"Management Hôtelier","duree":"3 mois","organisme":"IHEC Carthage / IHET","cout":"800 TND","mode":"Présentiel"},
    "pédagogie":      {"formation":"Ingénierie Pédagogique","duree":"6 semaines","organisme":"CEFE Tunis","cout":"400 TND","mode":"Mixte"},
    "agriculture":    {"formation":"Agriculture Durable & Irrigation","duree":"2 mois","organisme":"INAT Tunis","cout":"Subventionné","mode":"Présentiel"},
    "BTP":            {"formation":"Gestion de Chantier BTP","duree":"2 mois","organisme":"ATFP Tunis","cout":"Subventionné","mode":"Présentiel"},
    "soins infirmiers":{"formation":"Management en Soins Infirmiers","duree":"3 mois","organisme":"INAS Tunis","cout":"700 TND","mode":"Présentiel"},
}

print(f'✅ Base de données : {len(JOB_OFFERS)} offres | {len(FORMATIONS_CATALOGUE)} formations répertoriées')

✅ Base de données : 26 offres | 24 formations répertoriées


## 🤖 Cellule 6 — 5 Agents Spécialisés (Parser, Matcher, Gap, Coach, Ethics)

In [6]:
# ═══ CHARGEMENT MODÈLES ═══════════════════════════════════════════════════════

print('Chargement des modèles...')
embedder = SentenceTransformer(EMBED_MODEL)
client   = InferenceClient()  # Token HF depuis env HF_TOKEN
print(f'✅ Embedder {EMBED_MODEL} chargé')


# ── Pré-calcul embeddings offres + index BM25 (RAG hybride) ─────────────────
offer_texts = [
    f"{o['title']} {o['sector']} {' '.join(o['skills_required'])} {o['description']}"
    for o in JOB_OFFERS
]
offer_embeddings = embedder.encode(offer_texts, normalize_embeddings=True)

# BM25 index (sparse retrieval)
tokenized_corpus = [t.lower().split() for t in offer_texts]
bm25_index = BM25Okapi(tokenized_corpus)

print(f'✅ RAG hybride : {len(JOB_OFFERS)} offres indexées (dense + BM25)')


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT 1 — PARSER AGENT (LLM + fallback regex + score confiance)
# ═══════════════════════════════════════════════════════════════════════════════

def agent_parser(cv_text: str) -> dict:
    """Extrait le profil structuré depuis le texte brut du CV.
    Retourne un dict profil + confidence_score."""

    PROMPT = f"""Tu es un expert RH tunisien. Extrais les informations suivantes du CV en JSON strict.
Réponds UNIQUEMENT avec le JSON, sans texte avant ni après.

Format attendu :
{{
  "nom": "",
  "metier_actuel": "",
  "domaine": "",
  "experience_annees": 0,
  "niveau_formation": "",
  "region": "",
  "langues": [],
  "hard_skills": [],
  "soft_skills": [],
  "outils": [],
  "certifications": []
}}

CV à analyser :
{cv_text[:2500]}"""

    confidence = 1.0
    profile = {}

    try:
        response = client.chat.completions.create(
            model=HF_MODEL_LLM,
            messages=[{"role":"user","content":PROMPT}],
            max_tokens=600, temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        # Nettoyer si le modèle a ajouté des backticks
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        profile = json.loads(raw)

        # Score confiance : nombre de champs non-vides
        filled = sum(1 for v in profile.values() if v and v != [] and v != 0)
        confidence = round(filled / 11, 2)

    except Exception as e:
        print(f'   ⚠️  LLM Parser échec ({e}) — fallback regex')
        confidence = 0.45
        profile = _fallback_regex_parser(cv_text)

    # Defaults
    profile.setdefault('nom', 'Candidat')
    profile.setdefault('metier_actuel', 'Non précisé')
    profile.setdefault('domaine', 'Non précisé')
    profile.setdefault('experience_annees', 0)
    profile.setdefault('niveau_formation', 'Non précisé')
    profile.setdefault('region', 'Tunis')
    profile.setdefault('langues', ['Français'])
    profile.setdefault('hard_skills', [])
    profile.setdefault('soft_skills', [])
    profile.setdefault('outils', [])
    profile.setdefault('certifications', [])

    # Expansion KG
    all_skills = profile['hard_skills'] + profile['outils']
    profile['kg_expanded'] = kg_expand_skills(all_skills)
    profile['confidence']  = confidence

    return profile


def _fallback_regex_parser(text: str) -> dict:
    """Parser regex robuste si LLM indisponible."""
    known_skills = [
        "Python","SQL","Java","JavaScript","React","Node.js","TypeScript",
        "Machine Learning","Deep Learning","NLP","TensorFlow","PyTorch",
        "pandas","scikit-learn","Power BI","Tableau","Excel","Docker",
        "Kubernetes","Git","Linux","AWS","Azure","Django","FastAPI",
        "mécanique","maintenance","électricité","automatisme","PLC","SCADA",
        "AutoCAD","SolidWorks","GMAO","comptabilité","audit","ERP","SAP",
        "Agile","Scrum","marketing digital","SEO","CRM","LangChain","RAG",
        "MLOps","CI/CD","REST API","PostgreSQL","MySQL","MongoDB",
    ]
    text_lower = text.lower()
    found_skills = [s for s in known_skills if s.lower() in text_lower]

    # Expérience
    exp_match = re.search(r'(\d+)\s*(?:ans?|années?|years?|an d\'exp)', text_lower)
    experience = int(exp_match.group(1)) if exp_match else 0

    # Région
    regions = ["Tunis","Sfax","Sousse","Bizerte","Nabeul","Ariana",
               "Gafsa","Kasserine","Sidi Bouzid","Jendouba"]
    region_found = next((r for r in regions if r.lower() in text_lower), "Tunis")

    # Nom (première ligne non vide)
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    nom = lines[0] if lines else 'Candidat'

    return {
        "nom": nom,
        "metier_actuel": "Extrait par regex",
        "domaine": "Non précisé",
        "experience_annees": experience,
        "niveau_formation": "Non précisé",
        "region": region_found,
        "langues": ["Français"],
        "hard_skills": found_skills[:15],
        "soft_skills": [],
        "outils": [],
        "certifications": []
    }


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT 2 — MATCHER AGENT (RAG hybride dense + BM25 + Scoring 5D)
# ═══════════════════════════════════════════════════════════════════════════════

def agent_matcher(profile: dict) -> list:
    """Retourne les TOP_K_OFFERS offres avec scores 5D."""

    # ── Requête candidat ─────────────────────────────────────────────────────
    all_skills = profile['hard_skills'] + profile['soft_skills'] + profile['outils'] + profile['kg_expanded']
    query_text = f"{profile['metier_actuel']} {profile['domaine']} {' '.join(all_skills)}"
    query_emb  = embedder.encode([query_text], normalize_embeddings=True)

    # ── Score dense (cosine similarity) ──────────────────────────────────────
    dense_scores = cosine_similarity(query_emb, offer_embeddings)[0]

    # ── Score sparse (BM25) ───────────────────────────────────────────────────
    tokens_query = query_text.lower().split()
    bm25_scores_raw = bm25_index.get_scores(tokens_query)
    # Normaliser BM25 en [0,1]
    bm25_max = bm25_scores_raw.max() if bm25_scores_raw.max() > 0 else 1
    bm25_scores = bm25_scores_raw / bm25_max

    # ── Score hybride ─────────────────────────────────────────────────────────
    hybrid_scores = 0.6 * dense_scores + 0.4 * bm25_scores

    # Top candidats
    top_indices = np.argsort(hybrid_scores)[::-1][:TOP_K_OFFERS * 3]

    cand_skills_lower = {s.lower() for s in all_skills}

    results = []
    for idx in top_indices:
        offer = JOB_OFFERS[idx]
        req   = offer['skills_required']

        # Score sémantique (dense seul, plus pur)
        s_sem = round(float(dense_scores[idx]) * 100, 1)

        # Score skills (exact match)
        matched = [s for s in req if s.lower() in cand_skills_lower]
        gaps    = [s for s in req if s.lower() not in cand_skills_lower]
        s_skills = round(len(matched) / len(req) * 100, 1) if req else 0

        # Score expérience
        exp_min = offer['experience_min']
        cand_exp = profile['experience_annees']
        if cand_exp >= exp_min:
            s_exp = 100.0
        elif cand_exp >= exp_min - 1:
            s_exp = 70.0
        else:
            s_exp = max(0, 100 - (exp_min - cand_exp) * 20)

        # Score KG
        s_kg = kg_score(all_skills, req)

        # Score contexte (NOUVEAU v4) : région + type contrat bonus
        s_ctx = 50.0  # base
        if profile['region'].lower() == offer['region'].lower():
            s_ctx += 30.0
        if 'CDI' in offer['type']:
            s_ctx += 20.0
        s_ctx = min(s_ctx, 100.0)

        # Score total 5D pondéré
        s_total = round(
            WEIGHTS['semantic']   * s_sem +
            WEIGHTS['skills']     * s_skills +
            WEIGHTS['experience'] * s_exp +
            WEIGHTS['kg']         * s_kg +
            WEIGHTS['context']    * s_ctx, 1
        )

        # Bonus certifications
        cert_bonus = sum(2 for c in profile.get('certifications', [])
                         if any(c.lower() in r.lower() for r in req))
        s_total = min(100.0, s_total + cert_bonus)

        results.append({
            'offer': offer,
            'score_total': s_total,
            'score_semantic': s_sem,
            'score_skills': s_skills,
            'score_experience': round(s_exp, 1),
            'score_kg': s_kg,
            'score_context': round(s_ctx, 1),
            'matched_req': matched,
            'gaps': gaps,
        })

    results.sort(key=lambda x: x['score_total'], reverse=True)
    return results[:TOP_K_OFFERS]


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT 3 — GAP ANALYST
# ═══════════════════════════════════════════════════════════════════════════════

def agent_gap_analyst(matches: list) -> dict:
    """Analyse les gaps et enrichit avec le catalogue formations."""
    gap_count = {}
    gap_offers = {}
    for m in matches:
        for g in m['gaps']:
            gap_count[g] = gap_count.get(g, 0) + 1
            gap_offers.setdefault(g, []).append(m['offer']['title'])

    sorted_gaps = sorted(gap_count.items(), key=lambda x: x[1], reverse=True)
    gaps_list   = []
    for skill, freq in sorted_gaps:
        if freq >= 2:
            crit = '🔴 Critique'
        elif freq == 1 and matches[0]['score_total'] < 65:
            crit = '🟡 Important'
        else:
            crit = '🟢 Optionnel'
        form = FORMATIONS_CATALOGUE.get(skill) or \
               next((v for k, v in FORMATIONS_CATALOGUE.items()
                     if k.lower() in skill.lower() or skill.lower() in k.lower()), None)
        gaps_list.append({
            'skill': skill, 'frequence': freq, 'criticite': crit,
            'offres': gap_offers[skill], 'formation': form
        })

    return {'total_gaps': len(sorted_gaps), 'gaps_prioritises': gaps_list}


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT 4 — CAREER COACH
# ═══════════════════════════════════════════════════════════════════════════════

def agent_career_coach(profile: dict, matches: list, gaps: dict) -> dict:
    """Génère conseil personnalisé + 3 parcours simulés."""
    top_score = matches[0]['score_total'] if matches else 0
    top_offer = matches[0]['offer']['title'] if matches else 'Aucune offre'
    crit_gaps = [g['skill'] for g in gaps['gaps_prioritises'] if '🔴' in g['criticite']]

    # Conseil
    if top_score >= 80:
        conseil = f"🎯 Excellent profil ! Votre score de {top_score}% pour '{top_offer}' est très compétitif. Postulez directement et préparez un entretien technique axé sur {', '.join(profile['hard_skills'][:3])}."
    elif top_score >= 60:
        conseil = f"👍 Bon profil avec marge de progression. Score {top_score}% pour '{top_offer}'. Renforcez : {', '.join(crit_gaps[:3]) or 'quelques compétences techniques'}. 3-6 mois de formation ciblée suffisent."
    else:
        conseil = f"🛠️ Profil en développement (score {top_score}%). Commencez par les formations critiques : {', '.join(crit_gaps[:3]) or 'compétences de base'}. Envisagez un stage ou projet personnel pour valider les acquis."

    # Parcours 1 : Rapide (1-2 formations prioritaires)
    p1_steps = []
    for g in gaps['gaps_prioritises'][:2]:
        if g['formation']:
            p1_steps.append(f"{g['formation']['formation']} ({g['formation']['duree']}) — {g['formation']['organisme']}")
    if not p1_steps:
        p1_steps = ["Pratiquer sur des projets réels", "Compléter son profil LinkedIn"]

    # Parcours 2 : Complet (3-4 formations)
    p2_steps = []
    for g in gaps['gaps_prioritises'][:4]:
        if g['formation']:
            p2_steps.append(f"{g['formation']['formation']} ({g['formation']['duree']})")
    p2_steps.append("Stage / projet freelance (1 mois)")

    # Parcours 3 : Reconversion / Expert
    p3_domain = matches[0]['offer']['sector'] if matches else 'Tech'
    p3_steps  = [
        f"Fondamentaux secteur {p3_domain} — MOOC ciblé (2 mois)",
        "Projet capstone démontrant la maîtrise",
        "Réseautage professionnel (LinkedIn, MeetUp Tunis)",
        "Certification reconnue dans le secteur"
    ]

    score_p1 = min(100, top_score + 12)
    score_p2 = min(100, top_score + 22)
    score_p3 = min(100, top_score + 35)

    return {
        'conseil': conseil,
        'parcours': [
            {'titre': '⚡ Parcours Express',   'duree_totale': '1-2 mois',  'score_final_estime': score_p1, 'roi': 'Rapide', 'etapes': p1_steps},
            {'titre': '📈 Parcours Complet',   'duree_totale': '3-5 mois',  'score_final_estime': score_p2, 'roi': 'Optimal', 'etapes': p2_steps},
            {'titre': '🚀 Parcours Expert',    'duree_totale': '6-12 mois', 'score_final_estime': score_p3, 'roi': 'Long terme', 'etapes': p3_steps},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT 5 — ETHICS / BIAS AGENT (NOUVEAU v4)
# ═══════════════════════════════════════════════════════════════════════════════

def agent_ethics(profile: dict, matches: list) -> dict:
    """Détecte les potentiels biais dans le matching et génère un rapport éthique."""
    alerts = []
    score_impact = 0.0

    # Biais régional : candidat région défavorisée vs offres concentrées à Tunis
    cand_region = profile.get('region', 'Tunis')
    is_defavorised = any(r.lower() in cand_region.lower() for r in REGIONS_DEFAVORISEES)
    tunis_offers = sum(1 for m in matches if m['offer']['region'] in ["Tunis","Ariana","Ben Arous"])

    if is_defavorised:
        alerts.append({
            'type': '⚠️ Biais Régional',
            'detail': f"Candidat de {cand_region} (région prioritaire). {tunis_offers}/{len(matches)} offres matchées sont à Tunis. "
                      f"Des offres locales sont disponibles dans la base — vérifiez les filtres de proximité.",
            'recommandation': "Prioriser les offres régionales et opportunités de télétravail. Communiquer les aides à la mobilité ANETI."
        })
        score_impact += 0.05  # Potentiel désavantage structurel

    # Biais expérience : sur-pénalisation débutants
    exp = profile.get('experience_annees', 0)
    if exp <= 1:
        high_exp_offers = sum(1 for m in matches if m['offer']['experience_min'] >= 3)
        if high_exp_offers > len(matches) // 2:
            alerts.append({
                'type': '⚠️ Biais Expérience',
                'detail': f"Profil junior ({exp} an(s)) : {high_exp_offers} offres exigent 3+ ans. Le score expérience pénalise structurellement les débutants diplômés.",
                'recommandation': "Inclure des offres stages/alternance/junior. Valoriser les projets académiques et compétences transversales."
            })
            score_impact += 0.08

    # Parité genre (proxy : si prénom/nom détectable)
    # Note : Sans données démographiques, on signale la vigilance systémique
    alerts.append({
        'type': 'ℹ️ Parité Genre',
        'detail': "Le système n'utilise ni le genre ni l'âge dans le scoring. Aucun champ discriminant collecté.",
        'recommandation': "Audit régulier des taux de matching par profil démographique recommandé (ANETI/Gouvernance)."
    })

    # Transparence scoring
    bias_score = min(100, round(score_impact * 100))
    niveau_biais = '🔴 Élevé' if bias_score >= 10 else ('🟡 Modéré' if bias_score >= 5 else '🟢 Faible')

    return {
        'niveau_biais': niveau_biais,
        'score_biais': bias_score,
        'alertes': alerts,
        'transparency_note': f"Scoring basé sur {len(WEIGHTS)} dimensions explicables. Aucune donnée sensible (genre, âge, ethnie) n'est collectée ni utilisée.",
        'gouvernance': "Conforme aux principes IA responsable — ANETI 2025. Audit humain recommandé pour score > 75%."
    }


print('✅ 5 Agents chargés : Parser · Matcher (RAG hybride) · Gap Analyst · Career Coach · Ethics Agent')

Chargement des modèles...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✅ Embedder intfloat/multilingual-e5-small chargé
✅ RAG hybride : 26 offres indexées (dense + BM25)
✅ 5 Agents chargés : Parser · Matcher (RAG hybride) · Gap Analyst · Career Coach · Ethics Agent


## 🔗 Cellule 7 — Orchestrateur & Pipeline Complet

In [7]:
# ═══ ORCHESTRATEUR v4 ══════════════════════════════════════════════════════════

def run_pipeline(source: str) -> dict:
    """
    Pipeline complet v4 :
    Ingestion → Parser → Matcher (RAG hybride) → Gap Analyst → Coach → Ethics
    """
    t0 = time.time()

    # 1. Ingestion multi-format
    cv_text = ingestion.extract(source)
    ingestion_info = ingestion.report()
    print(f'  📥 Ingestion : {ingestion_info}')

    # 2. Parser Agent
    profile = agent_parser(cv_text)
    print(f'  🤖 Parser   : {profile["nom"]} | {profile["metier_actuel"]} | '
          f'confiance={profile["confidence"]:.0%} | {len(profile["hard_skills"])} skills')

    # 3. Matcher Agent (RAG hybride)
    matches = agent_matcher(profile)
    print(f'  🏆 Matcher  : Top offre → {matches[0]["offer"]["title"]} ({matches[0]["score_total"]}%)')

    # 4. Gap Analyst
    gaps = agent_gap_analyst(matches)
    print(f'  🔍 Gaps     : {gaps["total_gaps"]} gaps identifiés')

    # 5. Career Coach
    coaching = agent_career_coach(profile, matches, gaps)
    print(f'  💬 Coach    : {coaching["parcours"][0]["titre"]} → score estimé {coaching["parcours"][0]["score_final_estime"]}%')

    # 6. Ethics Agent (NOUVEAU v4)
    ethics = agent_ethics(profile, matches)
    print(f'  ⚖️  Ethics  : {ethics["niveau_biais"]} | {len(ethics["alertes"])} alertes')

    elapsed = round(time.time() - t0, 2)
    print(f'  ✅ Pipeline terminé en {elapsed}s')

    return {
        'profile':        profile,
        'matches':        matches,
        'gaps':           gaps,
        'coach':          coaching,
        'ethics':         ethics,
        'ingestion_info': ingestion_info,
        'elapsed_sec':    elapsed,
    }


# ─── CVs de test ──────────────────────────────────────────────────────────────

CV_EXAMPLES = {
    "Data Scientist": """
Sarra Ben Ali — Data Scientist | Tunis
4 ans d'expérience en Machine Learning et NLP
Compétences : Python, pandas, scikit-learn, TensorFlow, SQL, Power BI
Outils : Docker, Git, Jupyter, MLflow
Formation : Master Data Science — ESPRIT 2021
Langues : Français, Anglais, Arabe
Certifications : AWS Certified Machine Learning Specialty
""",
    "Ingénieur Mécanique": """
Ahmed Trabelsi — Ingénieur Mécanique | Sfax
5 ans maintenance industrielle (secteur textile et agro-alimentaire)
Compétences : mécanique, maintenance, diagnostic, hydraulique, pneumatique
Outils : AutoCAD, SolidWorks, GMAO
Formation : Ingénieur génie mécanique — ENIS Sfax 2019
Langues : Français, Arabe
""",
    "Développeur Full Stack": """
Maryam Khelil — Développeuse Full Stack | Sousse
3 ans d'expérience web (startups offshore)
Compétences : JavaScript, React, Node.js, Python, REST API
Outils : Docker, Git, PostgreSQL, Linux
Formation : Licence Informatique — ISI Tunis 2021
Langues : Français, Anglais
""",
    "Comptable Débutant (région)": """
Khalil Farhat — Comptable Junior | Kasserine
1 an d'expérience (stage cabinet comptable local)
Compétences : comptabilité, Excel, facturation
Formation : Licence Comptabilité — Université de Kasserine 2024
Langues : Français, Arabe
""",
}

print('✅ Orchestrateur v4 prêt — 5 agents coordonnés')
print('   CVs de test disponibles :', list(CV_EXAMPLES.keys()))

✅ Orchestrateur v4 prêt — 5 agents coordonnés
   CVs de test disponibles : ['Data Scientist', 'Ingénieur Mécanique', 'Développeur Full Stack', 'Comptable Débutant (région)']


## 🧪 Cellule 8 — Test du Pipeline

In [8]:
# ─── Test avec un CV exemple ──────────────────────────────────────────────────
# Changez le nom pour tester un autre profil
test_cv = CV_EXAMPLES["Data Scientist"]

print('=' * 65)
print('🧪 TEST PIPELINE v4')
print('=' * 65)

result = run_pipeline(test_cv)

print('\n── PROFIL EXTRAIT ──')
p = result['profile']
print(f"  Nom          : {p['nom']}")
print(f"  Métier       : {p['metier_actuel']}")
print(f"  Expérience   : {p['experience_annees']} an(s)")
print(f"  Région       : {p['region']}")
print(f"  Hard skills  : {', '.join(p['hard_skills'][:8])}")
print(f"  KG étendu    : {', '.join(p['kg_expanded'][:6])}")
print(f"  Confiance LLM: {p['confidence']:.0%}")

print('\n── TOP OFFRES (5D scoring) ──')
for i, m in enumerate(result['matches'], 1):
    o = m['offer']
    print(f"  {i}. {o['title'][:30]:30s} | Total: {m['score_total']:5.1f}% "
          f"| Sem:{m['score_semantic']:4.1f} Skl:{m['score_skills']:4.1f} "
          f"Exp:{m['score_experience']:4.1f} KG:{m['score_kg']:4.1f} Ctx:{m['score_context']:4.1f}")

print('\n── GAPS PRIORITAIRES ──')
for g in result['gaps']['gaps_prioritises'][:5]:
    f = g.get('formation', {})
    f_str = f"{f['formation']} ({f['duree']})" if f else "Pas dans le catalogue"
    print(f"  {g['criticite']} {g['skill']:20s} → {f_str}")

print('\n── CONSEIL COACH ──')
print(f"  {result['coach']['conseil'][:120]}...")

print('\n── RAPPORT ÉTHIQUE ──')
eth = result['ethics']
print(f"  Niveau biais : {eth['niveau_biais']}")
for a in eth['alertes']:
    print(f"  {a['type']} : {a['detail'][:80]}...")
print(f"  Gouvernance  : {eth['gouvernance'][:80]}...")

🧪 TEST PIPELINE v4
  📥 Ingestion : ✏️  Texte direct — 0 caractères
   ⚠️  LLM Parser échec (You must provide an api_key to work with auto API or log in with `hf auth login`.) — fallback regex
  🤖 Parser   : Sarra Ben Ali — Data Scientist | Tunis | Extrait par regex | confiance=45% | 11 skills
  🏆 Matcher  : Top offre → Data Scientist Senior (96.4%)
  🔍 Gaps     : 3 gaps identifiés
  💬 Coach    : ⚡ Parcours Express → score estimé 100%
  ⚖️  Ethics  : 🟢 Faible | 1 alertes
  ✅ Pipeline terminé en 0.14s

── PROFIL EXTRAIT ──
  Nom          : Sarra Ben Ali — Data Scientist | Tunis
  Métier       : Extrait par regex
  Expérience   : 4 an(s)
  Région       : Tunis
  Hard skills  : Python, SQL, Machine Learning, NLP, TensorFlow, pandas, scikit-learn, Power BI
  KG étendu    : contrôle de gestion, PyTorch, spaCy, MLflow, Kubernetes, Deep Learning
  Confiance LLM: 45%

── TOP OFFRES (5D scoring) ──
  1. Data Scientist Senior          | Total:  96.4% | Sem:89.6 Skl:100.0 Exp:100.0 KG:100.0 Ctx:10

## 🎨 Cellule 9 — Visualisations (Radar + Scores 5D + Gaps + Ethics)

In [9]:
def make_radar_chart(profile: dict) -> go.Figure:
    all_s = (profile.get('hard_skills',[]) + profile.get('soft_skills',[]) +
             profile.get('outils',[]) + profile.get('kg_expanded',[]))
    cand_lower = {s.lower() for s in all_s}
    cats = ["Data / IA","Tech / Industrie","Outils Numériques","Gestion / Mgmt","Communication"]
    keywords = {
        "Data / IA":         ["Python","SQL","pandas","Power BI","Machine Learning","TensorFlow","NLP","statistiques"],
        "Tech / Industrie":  ["mécanique","maintenance","électricité","automatisme","PLC","SCADA","GMAO","IOT"],
        "Outils Numériques": ["AutoCAD","SolidWorks","Git","Docker","JavaScript","React","Linux","Figma"],
        "Gestion / Mgmt":    ["gestion de projet","Agile","Scrum","leadership","audit","comptabilité","ERP"],
        "Communication":     ["communication","négociation","vente","recrutement","CRM","marketing digital"],
    }
    scores = [round(sum(1 for k in kws if k.lower() in cand_lower)/len(kws)*100) for kws in keywords.values()]
    fig = go.Figure(go.Scatterpolar(
        r=scores+[scores[0]], theta=cats+[cats[0]],
        fill='toself', fillcolor='rgba(56,138,221,0.15)',
        line=dict(color='#185FA5',width=2.5), marker=dict(size=8,color='#185FA5')
    ))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True,range=[0,100],ticksuffix='%')),
                      showlegend=False, height=350, title="Radar Employabilité")
    return fig


def make_scores_chart(matches: list) -> go.Figure:
    """Barres 5D — inclut score contexte (nouveau v4)."""
    titles = [m['offer']['title'][:28] for m in matches]
    dims   = ['score_semantic','score_skills','score_experience','score_kg','score_context']
    labels = ['Sémantique (35%)','Skills (30%)','Expérience (10%)','KG (15%)','Contexte (10%)']
    colors = ['#378ADD','#1D9E75','#EF9F27','#7F77DD','#E05252']
    fig = go.Figure()
    for dim, label, color in zip(dims, labels, colors):
        fig.add_trace(go.Bar(name=label, y=titles, x=[m[dim] for m in matches],
                             orientation='h', marker_color=color))
    fig.update_layout(barmode='group', height=380, title="Scores 5D par offre",
                      xaxis=dict(title='Score (%)', range=[0,100]),
                      legend=dict(orientation='h',yanchor='bottom',y=1.02,x=0),
                      margin=dict(l=10,r=10,t=90,b=10))
    return fig


def make_gaps_chart(gaps_analysis: dict) -> go.Figure:
    gaps = gaps_analysis.get('gaps_prioritises', [])
    if not gaps: return go.Figure()
    skills = [g['skill'] for g in gaps]
    freqs  = [g['frequence'] for g in gaps]
    colors = ['#E53E3E' if '🔴' in g['criticite'] else
              '#D69E2E' if '🟡' in g['criticite'] else '#38A169' for g in gaps]
    fig = go.Figure(go.Bar(y=skills[::-1], x=freqs[::-1], orientation='h',
                           marker_color=colors[::-1],
                           text=[f"{f} offres" for f in freqs[::-1]],
                           textposition='auto'))
    fig.update_layout(title="Gaps — Fréquence dans top offres",
                      xaxis=dict(title='Offres impactées'),
                      height=280, margin=dict(l=10,r=10,t=50,b=10))
    return fig


def make_ethics_chart(ethics: dict) -> go.Figure:
    """Gauge du niveau de biais éthique — NOUVEAU v4."""
    score = ethics.get('score_biais', 0)
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score,
        title={'text': "Score de Biais Détecté (%)"},
        gauge={
            'axis': {'range': [0, 25]},
            'bar': {'color': "#E53E3E" if score >= 10 else "#D69E2E" if score >= 5 else "#38A169"},
            'steps': [
                {'range': [0, 5],  'color': '#C6F6D5'},
                {'range': [5, 10], 'color': '#FEFCBF'},
                {'range': [10, 25],'color': '#FED7D7'},
            ],
        }
    ))
    fig.update_layout(height=250, margin=dict(l=20,r=20,t=60,b=20))
    return fig


# Test
make_radar_chart(result['profile']).show()
make_scores_chart(result['matches']).show()
make_gaps_chart(result['gaps']).show()
make_ethics_chart(result['ethics']).show()
print('✅ 4 visualisations générées (Radar + 5D Scores + Gaps + Ethics Gauge)')

✅ 4 visualisations générées (Radar + 5D Scores + Gaps + Ethics Gauge)


## 🖥️ Cellule 10 — Interface Gradio v4 (+ onglet Éthique + historique session)

In [10]:
# ═══ INTERFACE GRADIO v4 ══════════════════════════════════════════════════════

# Historique session (stocké en mémoire)
_session_history = []


def gradio_analyze(cv_file, cv_text_input, progress=gr.Progress()):
    progress(0.1, desc="📥 Lecture du CV...")
    source = cv_file.name if cv_file is not None else (cv_text_input or '').strip()
    if not source:
        err = "❌ Uploadez un fichier OU saisissez votre CV en texte."
        return [err, None, None, None, None, "—","—","—","—", err,err,err,err]

    try:
        progress(0.25, desc="🔍 Extraction et parsing...")
        result = run_pipeline(source)
        progress(0.85, desc="📊 Génération dashboard...")

        profile  = result['profile']
        matches  = result['matches']
        gaps     = result['gaps']
        coaching = result['coach']
        ethics   = result['ethics']

        # Ajouter à l'historique session
        _session_history.append({
            'nom': profile['nom'], 'score': matches[0]['score_total'],
            'offre': matches[0]['offer']['title'], 'biais': ethics['niveau_biais']
        })

        # ── Onglet Profil ─────────────────────────────────────────────────────
        conf_icon = '🟢' if profile['confidence'] >= 0.8 else ('🟡' if profile['confidence'] >= 0.5 else '🔴')
        profil_md = f"""
## 👤 Profil Extrait — Confiance LLM : {conf_icon} {profile['confidence']:.0%}
| Champ | Valeur |
|---|---|
| **Nom** | {profile.get('nom','—')} |
| **Métier** | {profile.get('metier_actuel','—')} |
| **Domaine** | {profile.get('domaine','—')} |
| **Expérience** | {profile.get('experience_annees',0)} an(s) |
| **Formation** | {profile.get('niveau_formation','—')} |
| **Région** | {profile.get('region','—')} |
| **Langues** | {', '.join(profile.get('langues',['—']))} |

**🔧 Hard Skills :** {', '.join(profile.get('hard_skills',['—']))}
**💼 Soft Skills :** {', '.join(profile.get('soft_skills',['—'])) or '—'}
**🛠️ Outils :** {', '.join(profile.get('outils',['—'])) or '—'}
**🏅 Certifications :** {', '.join(profile.get('certifications',[])) or 'Aucune'}
**🕸️ Compétences transférables (KG) :** {', '.join(profile.get('kg_expanded',[])[:8]) or '—'}

---
📥 **Source :** {result['ingestion_info']}
⏱️ **Temps :** {result['elapsed_sec']}s
"""

        # ── Onglet Offres ─────────────────────────────────────────────────────
        medals = ['🥇','🥈','🥉','4️⃣','5️⃣']
        offers_md = "## 🏆 Top Offres — Scoring 5D\n\n"
        for i, m in enumerate(matches, 1):
            o   = m['offer']
            bar = '█' * int(m['score_total']//10) + '░' * (10 - int(m['score_total']//10))
            offers_md += f"""
### {medals[i-1]} {o['title']} — {o['company']}
**Score : {m['score_total']}%** `{bar}`
📍 {o['region']} | 💼 {o['type']} | 💰 {o['salaire_range']} | 🏭 {o['sector']}

| Sem. | Skills | Exp. | KG | Contexte |
|------|--------|------|----|----------|
| {m['score_semantic']}% | {m['score_skills']}% | {m['score_experience']}% | {m['score_kg']}% | {m['score_context']}% |

✅ **Matchés :** {', '.join(m['matched_req']) or 'Aucun'}
❌ **Gaps :** {', '.join(m['gaps']) or 'Aucun 🎉'}
📝 {o['description']}\n---"""

        # ── Onglet Gaps ────────────────────────────────────────────────────────
        gaps_md = f"## 🔍 Gaps ({gaps['total_gaps']} compétences)\n\n"
        for g in gaps['gaps_prioritises']:
            f = g.get('formation', {})
            f_block = (f"\n> 🎓 **{f['formation']}** | {f['duree']} | {f['organisme']} | {f['cout']} | {f['mode']}"
                       if f else "\n> ℹ️ Recherche autonome recommandée")
            gaps_md += f"### {g['criticite']} — {g['skill']}\nPrésent dans **{g['frequence']}** offres{f_block}\n\n---\n"

        # ── Onglet Coach ───────────────────────────────────────────────────────
        coach_md = f"## 💬 Conseil\n\n{coaching['conseil']}\n\n---\n## 🗺️ Parcours de Carrière\n\n"
        for parc in coaching['parcours']:
            coach_md += f"### {parc['titre']}\n⏱️ {parc['duree_totale']} | 📈 Score estimé : **{parc['score_final_estime']}%** | 💡 ROI : {parc['roi']}\n\n"
            for e in parc['etapes']: coach_md += f"- {e}\n"
            coach_md += "\n---\n"

        # ── Onglet Éthique (NOUVEAU v4) ────────────────────────────────────────
        eth = result['ethics']
        ethics_md = f"## ⚖️ Rapport Éthique & Gouvernance\n\n"
        ethics_md += f"**Niveau de biais détecté :** {eth['niveau_biais']} ({eth['score_biais']}%)\n\n"
        ethics_md += f"**Transparence :** {eth['transparency_note']}\n\n"
        ethics_md += f"**Gouvernance :** {eth['gouvernance']}\n\n---\n"
        for a in eth['alertes']:
            ethics_md += f"### {a['type']}\n**Détail :** {a['detail']}\n\n**Recommandation :** {a['recommandation']}\n\n---\n"

        # KPI cards
        kpi_score = f"🎯 {matches[0]['score_total']}%"
        kpi_offer = matches[0]['offer']['title'][:30]
        kpi_gaps  = str(gaps['total_gaps'])
        kpi_level = ('🟢 Prêt' if matches[0]['score_total'] >= 80 else
                     '🟡 Presque' if matches[0]['score_total'] >= 60 else '🔴 Formation')

        radar_fig  = make_radar_chart(profile)
        scores_fig = make_scores_chart(matches)
        gaps_fig   = make_gaps_chart(gaps)
        ethics_fig = make_ethics_chart(ethics)

        progress(1.0, desc="✅ Terminé !")
        return [profil_md, radar_fig, scores_fig, gaps_fig, ethics_fig,
                kpi_score, kpi_offer, kpi_gaps, kpi_level,
                offers_md, gaps_md, coach_md, ethics_md]

    except Exception as e:
        import traceback
        err = f"❌ Erreur : {e}\n```\n{traceback.format_exc()}\n```"
        return [err, None, None, None, None, "—","—","—","—", err,err,err,err]


# ─── Construction interface ───────────────────────────────────────────────────

CSS = """
.kpi-card { background: #f0f4ff; border-radius: 12px; padding: 16px; text-align: center; }
.kpi-value { font-size: 2em; font-weight: bold; color: #185FA5; }
.kpi-label { font-size: 0.9em; color: #666; margin-top: 4px; }
.upload-box { border: 2px dashed #185FA5 !important; border-radius: 12px; }
.ethics-panel { background: #fff8e1; border-left: 4px solid #F6AD55; padding: 12px; border-radius: 8px; }
"""

with gr.Blocks(title="🧠 Mahara-Match AI v4", theme=gr.themes.Soft(), css=CSS) as demo:

    gr.Markdown("""
    # 🧠 Mahara-Match AI — Advanced PoC **v4.0**
    ### Matching intelligent CV ↔ Offres · 5 Agents · RAG Hybride · Scoring 5D · Gouvernance Éthique
    > 🆕 **v4** : +Ethics Agent · RAG hybride BM25+dense · Score contexte · Confiance LLM · +10 offres
    """)

    with gr.Row():
        with gr.Column(scale=4):
            with gr.Tab("📁 Upload Fichier"):
                cv_file = gr.File(label="Déposez votre CV",
                                  file_types=[".pdf",".docx",".doc",".jpg",".jpeg",".png",".txt"],
                                  elem_classes=["upload-box"])
                gr.Markdown("**Formats :** PDF · PDF scanné (OCR) · DOCX · JPG/PNG · TXT")
            with gr.Tab("✏️ Saisie Texte"):
                cv_text_input = gr.Textbox(label="Collez votre CV",
                                           placeholder="Prénom Nom\nPoste — X ans\nCompétences : Python, SQL...",
                                           lines=15, max_lines=30)

        with gr.Column(scale=1):
            analyze_btn = gr.Button("🚀 Analyser", variant="primary", size="lg")
            gr.Markdown("### 📊 KPIs")
            kpi_score = gr.Textbox(label="🎯 Meilleur score", interactive=False)
            kpi_offer = gr.Textbox(label="🏆 Offre n°1",      interactive=False)
            kpi_gaps  = gr.Textbox(label="❌ Gaps détectés",  interactive=False)
            kpi_level = gr.Textbox(label="🚦 Employabilité",  interactive=False)

    gr.Markdown("### ⚡ Exemples rapides")
    with gr.Row():
        for name, cv in CV_EXAMPLES.items():
            gr.Button(f"📋 {name}").click(fn=lambda c=cv: c, outputs=cv_text_input)

    gr.Markdown("---")
    with gr.Tabs():
        with gr.Tab("👤 Profil"):
            profil_out = gr.Markdown()
            radar_out  = gr.Plot(label="Radar Employabilité")
        with gr.Tab("🏆 Offres"):
            offers_out  = gr.Markdown()
            scores_out  = gr.Plot(label="Scores 5D")
        with gr.Tab("🔍 Gaps & Formations"):
            gaps_out  = gr.Markdown()
            gaps_plot = gr.Plot(label="Gaps — Fréquence")
        with gr.Tab("💬 Coach & Parcours"):
            coach_out = gr.Markdown()
        with gr.Tab("⚖️ Éthique & Gouvernance"):
            ethics_out  = gr.Markdown()
            ethics_plot = gr.Plot(label="Jauge de Biais")

    analyze_btn.click(
        fn=gradio_analyze,
        inputs=[cv_file, cv_text_input],
        outputs=[
            profil_out, radar_out, scores_out, gaps_plot, ethics_plot,
            kpi_score, kpi_offer, kpi_gaps, kpi_level,
            offers_out, gaps_out, coach_out, ethics_out
        ]
    )

    gr.Markdown("""
    ---
    *Mahara-Match AI v4.0 — PoC académique | ANETI Tunisie 2025-2026 | 5 Agents · RAG Hybride · Gouvernance IA Éthique*
    """)

demo.launch(share=True)
print('🚀 Interface Gradio v4 lancée')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6119bb203c34adc07c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🚀 Interface Gradio v4 lancée


## 📋 Cellule 11 — Export JSON + Rapport Markdown

In [11]:
def export_json(result: dict, path: str = "mahara_match_v4_result.json"):
    """Export JSON complet (compatible API ANETI)."""
    export = {
        "version": "v4.0",
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "ingestion": result['ingestion_info'],
        "elapsed_sec": result['elapsed_sec'],
        "profile": {k: v for k, v in result['profile'].items() if k != 'kg_scores'},
        "top_matches": [
            {
                "rank": i+1,
                "offer_id": m['offer']['id'],
                "offer_title": m['offer']['title'],
                "company": m['offer']['company'],
                "sector": m['offer']['sector'],
                "region": m['offer']['region'],
                "salaire_range": m['offer']['salaire_range'],
                "score_total": m['score_total'],
                "scores_5d": {
                    "semantic": m['score_semantic'],
                    "skills": m['score_skills'],
                    "experience": m['score_experience'],
                    "kg": m['score_kg'],
                    "context": m['score_context'],
                },
                "matched_skills": m['matched_req'],
                "gaps": m['gaps'],
            }
            for i, m in enumerate(result['matches'])
        ],
        "gaps_analysis": result['gaps'],
        "coaching": result['coach'],
        "ethics_report": result['ethics'],
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(export, f, ensure_ascii=False, indent=2)
    print(f'✅ Export JSON : {path} ({os.path.getsize(path):,} octets)')
    return export


def export_markdown_report(result: dict, path: str = "mahara_match_v4_rapport.md"):
    """Génère un rapport Markdown lisible (NOUVEAU v4)."""
    p = result['profile']
    m0 = result['matches'][0]
    eth = result['ethics']

    md = f"""# 📋 Rapport Mahara-Match AI v4.0
**Date :** {time.strftime('%d/%m/%Y %H:%M')}
**Candidat :** {p['nom']} — {p['metier_actuel']}
**Région :** {p['region']} | **Expérience :** {p['experience_annees']} an(s)

---

## 🏆 Meilleure Offre
**{m0['offer']['title']}** — {m0['offer']['company']}
Score global : **{m0['score_total']}%**
Sémantique: {m0['score_semantic']}% | Skills: {m0['score_skills']}% | Exp: {m0['score_experience']}% | KG: {m0['score_kg']}% | Contexte: {m0['score_context']}%

---

## 🔍 Gaps Critiques
"""
    for g in result['gaps']['gaps_prioritises']:
        if '🔴' in g['criticite']:
            f = g.get('formation', {})
            f_str = f"{f['formation']} ({f['duree']}, {f['organisme']})" if f else "À rechercher"
            md += f"- **{g['skill']}** → {f_str}\n"

    md += f"""

## 💬 Conseil Coach
{result['coach']['conseil']}

---

## ⚖️ Éthique
Niveau biais : {eth['niveau_biais']}
{eth['transparency_note']}

---
*Mahara-Match AI v4.0 | ANETI Tunisie*
"""
    with open(path, 'w', encoding='utf-8') as f:
        f.write(md)
    print(f'✅ Rapport Markdown : {path}')
    return path


# Exporter le résultat du test
exported = export_json(result)
export_markdown_report(result)

print("\n📄 Aperçu :")
print(json.dumps({
    "version":    exported['version'],
    "candidat":   exported['profile']['nom'],
    "top_offre":  exported['top_matches'][0]['offer_title'],
    "top_score":  exported['top_matches'][0]['score_total'],
    "n_gaps":     exported['gaps_analysis']['total_gaps'],
    "biais":      exported['ethics_report']['niveau_biais'],
}, ensure_ascii=False, indent=2))

✅ Export JSON : mahara_match_v4_result.json (6,535 octets)
✅ Rapport Markdown : mahara_match_v4_rapport.md

📄 Aperçu :
{
  "version": "v4.0",
  "candidat": "Sarra Ben Ali — Data Scientist | Tunis",
  "top_offre": "Data Scientist Senior",
  "top_score": 96.4,
  "n_gaps": 3,
  "biais": "🟢 Faible"
}
